## NBA Points Predictor - Bayesian Live Betting Board
**- Owner:** Alex

**- Status:** Live exploration

**- Last Updated:** 11/1/26

### Workflow

1. Load recent silver gamelogs and the newest multi-book points lines.
2. Fit the empirical-Bayes mean model on historical appearances (simplex 5/10/20 baseline + shared contextual slopes + partially pooled player intercept).
3. Build tonight's live feature rows:
   - opponent / home from `findOpp`
   - `opp_def_rating_prior` from that opponent's prior 10 completed games
   - team spread and game total from Pinnacle team markets
   - starting `1/0` from RotoWire posted fives (`starter_surprise = starting - start_rate`)
4. Score every listed points line and rank top EV overs/unders.

Validation cells below remain for chronological research; the live board is the operational output.


### Load Data and Current Point Lines

In [8]:
from pathlib import Path
import sys


import numpy as np
import pandas as pd
from IPython.display import display


ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
SEASONS = ("2022-23", "2023-24", "2024-25", "2025-26", "2026-27")


# Exclusive game-date cutoff. Match board slate_date (Pacific calendar day).
# Change this for historical forecasts.
AS_OF = pd.Timestamp(
    pd.Timestamp.now(tz="America/Los_Angeles").date(),
    tz="UTC",
)
SEED = 20261032


pd.set_option("display.max_columns", None)


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))




chunks = []


for season in SEASONS:
    path = (
        ROOT / "data" / "silver" / "nba" / season
        / "regular_season" / "player_gamelogs.parquet"
    )


    if not path.exists():
        print(f"Skipping missing season: {season}")
        continue


    chunk = pd.read_parquet(path)
    chunk["season_year"] = season
    chunks.append(chunk)


if not chunks:
    raise FileNotFoundError("No regular-season gamelog files found.")


df = pd.concat(chunks, ignore_index=True)


required = [
    "player_id", "player_name", "game_id", "game_date",
    "team_id", "opp_team_id", "matchup",
    "pts", "minutes", "usg_pct", "start_position",
    "fga", "fta",
    "team_def_rating",
    "player_team_spread", "game_total",
]


missing = sorted(set(required) - set(df.columns))


if missing:
    raise ValueError(f"Missing columns: {missing}")


df["game_date"] = pd.to_datetime(
    df["game_date"], utc=True, errors="raise"
)
df["player_id"] = pd.to_numeric(
    df["player_id"], errors="raise"
).astype("Int64")
df["team_id"] = pd.to_numeric(df["team_id"], errors="raise").astype("Int64")
df["opp_team_id"] = pd.to_numeric(
    df["opp_team_id"], errors="raise"
).astype("Int64")
df["game_id"] = df["game_id"].astype("string")


if df[
    ["player_id", "game_id", "game_date", "team_id", "opp_team_id"]
].isna().any().any():
    raise ValueError("Missing IDs or game dates.")


for column in (
    "pts", "minutes", "usg_pct", "fga", "fta",
    "team_def_rating", "player_team_spread", "game_total",
):
    df[column] = pd.to_numeric(
        df[column], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)


df = df.loc[df["game_date"].lt(AS_OF)].copy()


keys = ["player_id", "game_id"]
duplicates = df.loc[df.duplicated(keys, keep=False)]


check_cols = [
    "game_date", "season_year", "pts",
    "minutes", "usg_pct", "start_position",
    "team_id", "opp_team_id", "fga", "fta",
    "team_def_rating", "player_team_spread", "game_total",
]


if not duplicates.empty:
    conflicts = (
        duplicates.groupby(keys)[check_cols]
        .nunique(dropna=False)
        .gt(1)
    )


    if conflicts.any().any():
        raise ValueError("Conflicting duplicate player-game records.")


    print(f"Removing {int(df.duplicated(keys).sum())} duplicate rows.")


df = (
    df.drop_duplicates(keys)
    .sort_values(["player_id", "game_date", "game_id"])
    .reset_index(drop=True)
)


print(f"{len(df):,} rows with game dates before {AS_OF.date()}.")
print(
    "Spread/total coverage:",
    f"player_team_spread {df['player_team_spread'].notna().mean():.1%},",
    f"game_total {df['game_total'].notna().mean():.1%}.",
)
print(
    "Line provenance: silver `player_team_spread` / `game_total` come from "
    "Rotowire enrichment in the silver merge (closing/game lines archived "
    "by game date). They are historical settled lines, not a live pregame "
    "snapshot feed. Upcoming predictions must supply actual pregame inputs."
)
display(df["usg_pct"].describe())
display(df["start_position"].value_counts(dropna=False))




from src.live.latest_points import latest_points_markets


markets, events, audit = latest_points_markets(ROOT / "data" / "props")


for book, source in audit["source_files"].items():
    n = int(markets["book"].eq(book).sum())
    print(f"{book}: {source} ({n} points lines)")


if audit["missing_books"]:
    print("Missing:", ", ".join(audit["missing_books"]))


display(markets.head())


107,242 rows with game dates before 2026-11-01.
Spread/total coverage: player_team_spread 98.2%, game_total 98.2%.
Line provenance: silver `player_team_spread` / `game_total` come from Rotowire enrichment in the silver merge (closing/game lines archived by game date). They are historical settled lines, not a live pregame snapshot feed. Upcoming predictions must supply actual pregame inputs.


count    107242.000000
mean          0.181432
std           0.087516
min           0.000000
25%           0.125000
50%           0.176000
75%           0.235000
max           1.000000
Name: usg_pct, dtype: float64

start_position
     57162
F    20032
G    20032
C    10016
Name: count, dtype: int64

prizepicks: prizepicks_nba_2026-11-01_181500.json (111 points lines)
underdog: underdog_nba_2026-11-01_181500.json (99 points lines)
novig: novig_nba_2026-11-01_181501_props.json (102 points lines)
prophetx: prophetx_nba_2026-11-01_181501_props.json (100 points lines)
pinnacle: pinnacle_nba_2026-11-01_181500_props.json (0 points lines)
fanduel: fanduel_nba_2026-11-01_181502.json (103 points lines)
draftkings: draftkings_nba_2026-11-01_181501.json (0 points lines)


,slate_date,book,player_name,line,over_american,under_american
0,2026-11-01,prizepicks,Shai Gilgeous-Alexander,29.5,-119.0,-119.0
1,2026-11-01,prizepicks,Donovan Mitchell,22.5,-119.0,-119.0
2,2026-11-01,prizepicks,James Harden,18.5,-119.0,-119.0
3,2026-11-01,prizepicks,Ajay Mitchell,17.5,-119.0,-119.0
4,2026-11-01,prizepicks,Evan Mobley,15.5,-119.0,-119.0


### Construct Historical Features

Build rolling predictors on the full played-appearance history, then apply model eligibility. Target-game values are excluded from every rolling window. Attempt missingness is preserved (no zero fill).


In [9]:
# Game-environment and market features on the loaded slate.
# is_home: schedule matchup token ("vs." home, "@" away).
# Spreads/totals: silver Rotowire-backed columns (see loader note).


matchup = df["matchup"].astype("string")
is_home = matchup.str.contains(r"\bvs\.", regex=True)
is_away = matchup.str.contains(r"\s@\s", regex=True)


if bool((is_home & is_away).any()) or bool((~is_home & ~is_away).any()):
    raise ValueError("matchup home/away tokens are incomplete or overlapping.")


df["is_home"] = is_home.astype(float)
df["team_spread_pregame"] = df["player_team_spread"]
df["game_total_pregame"] = df["game_total"]


# Negative player_team_spread denotes the favorite (team expected to win).
df["implied_team_total_pregame"] = (
    df["game_total_pregame"] - df["team_spread_pregame"]
) / 2.0


print(
    "Home rate:",
    f"{df['is_home'].mean():.3f};",
    "implied total vs spread corr:",
    f"{df['team_spread_pregame'].corr(df['implied_team_total_pregame']):.3f}",
)


Home rate: 0.501; implied total vs spread corr: -0.673


In [10]:
USAGE_UNITS = "fraction"
MISSING_START_POSITION_IS_BENCH = False

WINDOWS = (5, 10, 20)


played = df["minutes"].gt(0) & df["pts"].notna()
hist = df.loc[played].copy().reset_index(drop=True)


if hist.empty:
    raise ValueError("No played games with valid points.")


if (
    (hist["pts"] < 0).any()
    or not np.allclose(hist["pts"], np.round(hist["pts"]))
):
    raise ValueError("Points must be nonnegative integers.")


if USAGE_UNITS not in ("fraction", "percent"):
    raise ValueError("USAGE_UNITS must be 'fraction' or 'percent'.")


hist["usage_pp"] = hist["usg_pct"] * (
    100.0 if USAGE_UNITS == "fraction" else 1.0
)


if hist["usage_pp"].dropna().median() > 100:
    raise ValueError(
        "Usage units appear incorrect; inspect the loader's schema."
    )


# Preserve attempt missingness; do not fill with zero.
for column in ("fga", "fta", "minutes", "usage_pp"):
    hist[column] = pd.to_numeric(hist[column], errors="coerce")


position = (
    hist["start_position"]
    .astype("string")
    .str.strip()
    .str.upper()
)


unrecognized = position.loc[
    position.notna() & ~position.isin(["", "G", "F", "C"])
].unique()


if len(unrecognized):
    raise ValueError(
        f"Unrecognized start-position labels: {list(unrecognized)}"
    )


hist["starter"] = np.nan
hist.loc[position.isin(["G", "F", "C"]), "starter"] = 1.0
hist.loc[position.eq("").fillna(False), "starter"] = 0.0


if MISSING_START_POSITION_IS_BENCH:
    hist.loc[position.isna(), "starter"] = 0.0


missing_starter = int(hist["starter"].isna().sum())
print(f"Played games with missing starter labels: {missing_starter:,}.")




def prior_mean(frame, column, window):
    return frame.groupby("player_id", sort=False)[column].transform(
        lambda values: values.shift(1).rolling(
            window, min_periods=window
        ).mean()
    )




# --- Team-game environment history (one row per team-game) ---
team_games = (
    hist[
        [
            "team_id", "game_id", "game_date",
            "team_def_rating", "is_home",
        ]
    ]
    .copy()
)


rating_conflicts = (
    team_games.groupby(["team_id", "game_id"])["team_def_rating"]
    .nunique(dropna=False)
    .gt(1)
    .any()
)
home_conflicts = (
    team_games.groupby(["team_id", "game_id"])["is_home"]
    .nunique(dropna=False)
    .gt(1)
    .any()
)


if rating_conflicts or home_conflicts:
    raise ValueError("Repeated team-game environment stats disagree.")


team_games = (
    team_games.drop_duplicates(["team_id", "game_id"])
    .sort_values(["team_id", "game_date", "game_id"])
    .reset_index(drop=True)
)


team_games["team_def_rating_prior_10"] = (
    team_games.groupby("team_id", sort=False)["team_def_rating"]
    .transform(
        lambda values: values.shift(1).rolling(10, min_periods=10).mean()
    )
)


prev_date = team_games.groupby("team_id", sort=False)["game_date"].shift(1)
day_gap = (
    team_games["game_date"].dt.normalize()
    - prev_date.dt.normalize()
).dt.days
team_games["back2back"] = day_gap.eq(1).astype(float)
team_games.loc[prev_date.isna(), "back2back"] = 0.0


hist = hist.merge(
    team_games[["team_id", "game_id", "back2back"]],
    on=["team_id", "game_id"],
    how="left",
    validate="many_to_one",
)


opp_prior = team_games[
    ["team_id", "game_id", "team_def_rating_prior_10"]
].rename(
    columns={
        "team_id": "opp_team_id",
        "team_def_rating_prior_10": "opp_def_rating_prior",
    }
)
hist = hist.merge(
    opp_prior,
    on=["game_id", "opp_team_id"],
    how="left",
    validate="many_to_one",
)




for window in WINDOWS:
    hist[f"p{window}"] = prior_mean(hist, "pts", window)


hist["usage_trend"] = (
    prior_mean(hist, "usage_pp", 5)
    - prior_mean(hist, "usage_pp", 20)
)
hist["minutes_trend"] = (
    prior_mean(hist, "minutes", 5)
    - prior_mean(hist, "minutes", 20)
)
hist["fga_trend"] = (
    prior_mean(hist, "fga", 5)
    - prior_mean(hist, "fga", 20)
)
hist["fta_trend"] = (
    prior_mean(hist, "fta", 5)
    - prior_mean(hist, "fta", 20)
)


hist["start_rate"] = prior_mean(hist, "starter", 20)
# starter_surprise uses the current-game starter label minus prior start rate.
hist["starter_surprise"] = hist["starter"] - hist["start_rate"]


BASE_CONTEXT_COLS = (
    "starter_surprise",
    "is_home",
    "back2back",
    "opp_def_rating_prior",
    "team_spread_pregame",
    "implied_team_total_pregame",
)
USAGE_MINUTES_TRENDS = ("usage_trend", "minutes_trend")
ATTEMPT_MINUTES_TRENDS = ("fga_trend", "fta_trend", "minutes_trend")
BOTH_TRENDS = (
    "usage_trend", "minutes_trend", "fga_trend", "fta_trend",
)


FEATURE_SETS = {
    "usage_minutes": BASE_CONTEXT_COLS + USAGE_MINUTES_TRENDS,
    "attempt_minutes": BASE_CONTEXT_COLS + ATTEMPT_MINUTES_TRENDS,
    "both_trends": BASE_CONTEXT_COLS + BOTH_TRENDS,
}


# Default specification matches both_trends (spread + implied team total).
CONTEXT_FEATURE_COLS = FEATURE_SETS["both_trends"]


window_needed = ["p5", "p10", "p20", "start_rate"]
base_needed = list(window_needed) + list(BASE_CONTEXT_COLS)


def complete_mask(frame, columns):
    return np.isfinite(
        frame[list(columns)].to_numpy(dtype=float, na_value=np.nan)
    ).all(axis=1)


eligibility_counts = {
    "played_history": len(hist),
    "windows_start_rate": int(complete_mask(hist, window_needed).sum()),
    "base_context": int(complete_mask(hist, base_needed).sum()),
}


for name, cols in FEATURE_SETS.items():
    needed = list(window_needed) + list(cols)
    eligibility_counts[name] = int(complete_mask(hist, needed).sum())


print("Eligibility counts after successive feature requirements:")
display(pd.Series(eligibility_counts).to_frame("n_rows"))


common_needed = list(window_needed) + list(
    dict.fromkeys(
        column
        for cols in FEATURE_SETS.values()
        for column in cols
    )
)
common_mask = complete_mask(hist, common_needed)
eligible = hist.loc[common_mask].copy()


print(
    f"Played history: {len(hist):,}; "
    f"rows with all compared feature sets: {len(eligible):,}."
)
print(
    "Feature missingness (played history):",
    hist[common_needed].isna().sum().to_dict(),
)


if eligible.empty:
    raise ValueError(
        "No eligible games; check historical coverage "
        "and starter-label semantics."
    )




from src.pipeline.silver.positions import player_name_keys


key_to_ids = {}


for pid, name in (
    hist[["player_id", "player_name"]]
    .drop_duplicates()
    .itertuples(index=False)
):
    for key in player_name_keys(name):
        key_to_ids.setdefault(key, set()).add(int(pid))




def resolve_player_id(name):
    ids = set()


    for key in player_name_keys(name):
        ids.update(key_to_ids.get(key, set()))


    return next(iter(ids)) if len(ids) == 1 else pd.NA




board = markets.copy()
board["player_id"] = (
    board["player_name"].map(resolve_player_id).astype("Int64")
)


print(f"Matched board players: {board['player_id'].nunique()}.")


Played games with missing starter labels: 0.
Eligibility counts after successive feature requirements:


,n_rows
played_history,107233
windows_start_rate,91522
base_context,89786
usage_minutes,89786
attempt_minutes,89786
both_trends,89786


Played history: 107,233; rows with all compared feature sets: 89,786.
Feature missingness (played history): {'p5': 4445, 'p10': 8420, 'p20': 15711, 'start_rate': 15711, 'starter_surprise': 15711, 'is_home': 0, 'back2back': 0, 'opp_def_rating_prior': 3285, 'team_spread_pregame': 1882, 'implied_team_total_pregame': 1905, 'usage_trend': 15711, 'minutes_trend': 15711, 'fga_trend': 15711, 'fta_trend': 15711}
Matched board players: 115.


### Fit the empirical-Bayes mean model with contextual Gaussian priors


In [11]:
import numpy as np
import pandas as pd


from src.models.points_empirical_bayes import (
    CONTEXT_COLS,
    PRIOR_SD_RAW,
    WINDOW_COLS,
    fit_points_model,
    observation_variance,
    predict_points,
    shared_noise_posterior_moments,
)


assert tuple(CONTEXT_FEATURE_COLS) == tuple(CONTEXT_COLS), (
    "Notebook default feature set drifted from src CONTEXT_COLS."
)


fitted_demo = fit_points_model(
    eligible,
    variance_mode="baseline",
    context_cols=CONTEXT_FEATURE_COLS,
)


print(
    "Variance mode:",
    fitted_demo["variance_mode"],
)
print(
    "Optimizer:",
    fitted_demo["optimizer_message"],
    f"(nfev={fitted_demo['optimizer_nfev']})",
)
print(
    "Boundary solutions:",
    fitted_demo["at_bound"] or "none",
)
print(
    "gamma0={:.3f}, gamma1={:.3f}, "
    "log(1+B) center={:.3f}, "
    "player-intercept prior sd={:.3f}".format(
        fitted_demo["gamma0"],
        fitted_demo["gamma1"],
        fitted_demo["log_b_center"],
        np.sqrt(fitted_demo["player_var"]),
    )
)
print(fitted_demo["uncertainty_approximation"])
print("Context columns:", fitted_demo["context_cols"])


baseline_grid = np.linspace(0, 40, 9)
sigma_grid = observation_variance(
    baseline_grid,
    fitted_demo["gamma0"],
    fitted_demo["gamma1"],
    fitted_demo["log_b_center"],
)
display(pd.DataFrame({
    "baseline_B": baseline_grid,
    "observation_sd": np.sqrt(sigma_grid),
}).round(3))


prior_table = pd.DataFrame({
    "coefficient": ["level", *fitted_demo["context_cols"]],
    "prior_sd_raw": [
        fitted_demo["prior_sd_raw"]["level"],
        *[
            fitted_demo["prior_sd_raw"][column]
            for column in fitted_demo["context_cols"]
        ],
    ],
    "train_scale": np.r_[1.0, fitted_demo["scale"]],
    "prior_sd_standardized": fitted_demo["prior_sd"],
    "posterior_mean_original": fitted_demo["beta_original"],
    "posterior_sd_original": np.sqrt(
        np.diag(fitted_demo["beta_cov_original"])
    ),
})
display(prior_table.round(3))


Variance mode: baseline
Optimizer: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH (nfev=136)
Boundary solutions: none
gamma0=3.505, gamma1=0.786, log(1+B) center=2.339, player-intercept prior sd=0.000
plug-in empirical Bayes for baseline weights and variance hyperparameters; posterior uncertainty for shared coefficients and player intercepts only
Context columns: ('starter_surprise', 'is_home', 'back2back', 'opp_def_rating_prior', 'team_spread_pregame', 'implied_team_total_pregame', 'usage_trend', 'minutes_trend', 'fga_trend', 'fta_trend')


,baseline_B,observation_sd
0,0.0,2.300
1,5.0,4.653
2,10.0,5.905
3,15.0,6.843
4,20.0,7.615
5,25.0,8.282
6,30.0,8.875
7,35.0,9.413
8,40.0,9.907


,coefficient,prior_sd_raw,train_scale,prior_sd_standardized,posterior_mean_original,posterior_sd_original
0,level,3.00,1.000,3.000,0.193,0.018
1,starter_surprise,5.00,0.320,1.598,3.238,0.060
2,is_home,2.00,0.500,1.000,0.225,0.038
3,back2back,2.00,0.379,0.757,0.278,0.048
4,opp_def_rating_prior,0.30,4.783,1.435,0.071,0.004
5,team_spread_pregame,0.15,7.911,1.187,0.020,0.003
6,implied_team_total_pregame,0.20,5.857,1.171,-0.029,0.004
7,usage_trend,0.50,2.778,1.389,-0.015,0.007
8,minutes_trend,0.75,3.650,2.737,0.039,0.008
9,fga_trend,1.00,1.750,1.750,-0.060,0.021


### Validate trend feature sets on identical chronological folds

Compare usage/minutes trends, FGA/FTA + minutes trends, and both together, holding the shared schedule/market context fixed. Report eligibility losses, RMSE, signed bias, predictive negative log score, and 80%/95% interval coverage and width. Prior sensitivity uses half/twice `PRIOR_SD_RAW` on development folds (all chronological folds except the latest test season).


In [12]:
from scipy.stats import norm


def start_probability(frame):
    return (
        20.0 * frame["start_rate"].to_numpy(dtype=float) + 1.0
    ) / 22.0




def normal_point_mass(actual, prediction):
    sd = np.sqrt(prediction["variance"])
    lower = (actual - 0.5 - prediction["mean"]) / sd
    upper = (actual + 0.5 - prediction["mean"]) / sd
    mass = np.where(
        lower >= 0,
        norm.sf(lower) - norm.sf(upper),
        norm.cdf(upper) - norm.cdf(lower),
    )
    return np.maximum(mass, np.finfo(float).tiny)




def mixture_quantile(q, start, bench, probability):
    lo = np.minimum(
        start["mean"] - 10 * np.sqrt(start["variance"]),
        bench["mean"] - 10 * np.sqrt(bench["variance"]),
    )
    hi = np.maximum(
        start["mean"] + 10 * np.sqrt(start["variance"]),
        bench["mean"] + 10 * np.sqrt(bench["variance"]),
    )
    for _ in range(45):
        mid = (lo + hi) / 2
        cdf = q * norm.cdf(
            mid, start["mean"], np.sqrt(start["variance"])
        )
        cdf += (1 - q) * norm.cdf(
            mid, bench["mean"], np.sqrt(bench["variance"])
        )
        lo = np.where(cdf < probability, mid, lo)
        hi = np.where(cdf < probability, hi, mid)
    return (lo + hi) / 2




def chronological_folds(frame):
    seasons = sorted(
        frame["season_year"].unique(),
        key=lambda value: int(value[:4]),
    )
    folds = []
    for index in range(2, len(seasons)):
        test = frame.loc[frame["season_year"].eq(seasons[index])].copy()
        train = frame.loc[frame["season_year"].isin(seasons[:index])].copy()
        train = train.loc[train["game_date"].lt(test["game_date"].min())]
        if train.empty or test.empty:
            continue
        folds.append((seasons[index], train, test))
    return folds




def evaluate_predictions(test, fitted_fold):
    start = predict_points(test, fitted_fold, starter_value=1.0)
    bench = predict_points(test, fitted_fold, starter_value=0.0)
    q = start_probability(test)
    mean = q * start["mean"] + (1 - q) * bench["mean"]
    variance = (
        q * start["variance"]
        + (1 - q) * bench["variance"]
        + q * (1 - q) * (start["mean"] - bench["mean"]) ** 2
    )
    actual = test["pts"].to_numpy(dtype=float)
    error = actual - mean
    mass = (
        q * normal_point_mass(actual, start)
        + (1 - q) * normal_point_mass(actual, bench)
    )
    row = {
        "n": len(test),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "bias_actual_minus_mean": float(np.mean(error)),
        "negative_log_score": float(-np.mean(np.log(mass))),
        "mean_predictive_sd": float(np.mean(np.sqrt(variance))),
    }
    for coverage in (0.80, 0.95):
        tail = (1 - coverage) / 2
        lower = np.floor(mixture_quantile(q, start, bench, tail) + 0.5)
        upper = np.floor(mixture_quantile(q, start, bench, 1 - tail) + 0.5)
        covered = (actual >= lower) & (actual <= upper)
        name = f"coverage_{int(coverage * 100)}"
        row[name] = float(covered.mean())
        row[f"width_{int(coverage * 100)}"] = float(np.mean(upper - lower))
    return row, {
        "mean": mean,
        "variance": variance,
        "error": error,
        "actual": actual,
    }




def evaluate_feature_sets(frame, feature_sets, variance_mode="baseline"):
    fold_rows = []
    for test_season, train, test in chronological_folds(frame):
        for set_name, cols in feature_sets.items():
            fitted_fold = fit_points_model(
                train,
                variance_mode=variance_mode,
                context_cols=cols,
            )
            metrics, _ = evaluate_predictions(test, fitted_fold)
            fold_rows.append({
                "feature_set": set_name,
                "test_season": test_season,
                "n_train": len(train),
                "gamma1": fitted_fold["gamma1"],
                "player_intercept_prior_sd": np.sqrt(fitted_fold["player_var"]),
                **metrics,
            })
    return pd.DataFrame(fold_rows)




eligibility_loss = pd.DataFrame([
    {
        "feature_set": name,
        "eligible_rows": eligibility_counts[name],
        "loss_vs_base_context": (
            eligibility_counts["base_context"] - eligibility_counts[name]
        ),
        "loss_vs_windows": (
            eligibility_counts["windows_start_rate"] - eligibility_counts[name]
        ),
    }
    for name in FEATURE_SETS
])
print("Eligibility losses from added contextual / trend features:")
display(eligibility_loss)


fold_table = evaluate_feature_sets(eligible, FEATURE_SETS)
print("Chronological fold metrics on identical eligible rows:")
display(fold_table.round(3))


summary = (
    fold_table.groupby("feature_set", sort=False)
    .agg(
        n=("n", "sum"),
        rmse=("rmse", "mean"),
        bias=("bias_actual_minus_mean", "mean"),
        nll=("negative_log_score", "mean"),
        coverage_80=("coverage_80", "mean"),
        width_80=("width_80", "mean"),
        coverage_95=("coverage_95", "mean"),
        width_95=("width_95", "mean"),
    )
    .reset_index()
)
print("Mean metrics across chronological folds:")
display(summary.round(3))


# Prior sensitivity on development folds (exclude latest test season).
dev_seasons = sorted(
    fold_table["test_season"].unique(),
    key=lambda value: int(value[:4]),
)
if len(dev_seasons) >= 2:
    holdout_season = dev_seasons[-1]
    sensitivity_rows = []
    for test_season, train, test in chronological_folds(eligible):
        if test_season == holdout_season:
            continue
        for scale in (0.5, 1.0, 2.0):
            fitted_fold = fit_points_model(
                train,
                variance_mode="baseline",
                context_cols=FEATURE_SETS["both_trends"],
                prior_sd_scale=scale,
            )
            metrics, _ = evaluate_predictions(test, fitted_fold)
            sensitivity_rows.append({
                "prior_sd_scale": scale,
                "test_season": test_season,
                **metrics,
            })
    sensitivity = pd.DataFrame(sensitivity_rows)
    print(
        f"Prior-SD sensitivity on development folds "
        f"(held out latest test season {holdout_season}):"
    )
    display(
        sensitivity.groupby("prior_sd_scale", sort=True)
        .agg(
            n=("n", "sum"),
            rmse=("rmse", "mean"),
            bias=("bias_actual_minus_mean", "mean"),
            nll=("negative_log_score", "mean"),
            coverage_80=("coverage_80", "mean"),
            coverage_95=("coverage_95", "mean"),
        )
        .reset_index()
        .round(3)
    )
else:
    print(
        "Fewer than two chronological test seasons; "
        "prior sensitivity could not reserve a holdout season."
    )
    sensitivity = pd.DataFrame()


validation_predictions = fold_table


Eligibility losses from added contextual / trend features:


,feature_set,eligible_rows,loss_vs_base_context,loss_vs_windows
0,usage_minutes,89786,0,1736
1,attempt_minutes,89786,0,1736
2,both_trends,89786,0,1736


Chronological fold metrics on identical eligible rows:


,feature_set,test_season,n_train,gamma1,player_intercept_prior_sd,n,rmse,bias_actual_minus_mean,negative_log_score,mean_predictive_sd,coverage_80,width_80,coverage_95,width_95
0,usage_minutes,2024-25,39394,0.820,0.000,24377,6.165,-0.056,3.192,5.956,0.852,15.265,0.959,23.350
1,attempt_minutes,2024-25,39394,0.819,0.000,24377,6.165,-0.053,3.192,5.956,0.852,15.260,0.959,23.347
2,both_trends,2024-25,39394,0.819,0.001,24377,6.165,-0.053,3.192,5.956,0.852,15.260,0.959,23.348
3,usage_minutes,2025-26,63771,0.784,0.000,24246,6.107,-0.164,3.180,6.002,0.854,15.379,0.962,23.527
4,attempt_minutes,2025-26,63771,0.783,0.000,24246,6.106,-0.164,3.179,6.002,0.854,15.381,0.962,23.528
5,both_trends,2025-26,63771,0.783,0.001,24246,6.106,-0.163,3.180,6.002,0.854,15.381,0.962,23.526
6,usage_minutes,2026-27,88017,0.784,0.001,1769,6.565,-1.847,3.242,6.260,0.821,16.034,0.970,24.525
7,attempt_minutes,2026-27,88017,0.784,0.001,1769,6.564,-1.851,3.242,6.259,0.822,16.040,0.971,24.551
8,both_trends,2026-27,88017,0.784,0.000,1769,6.564,-1.851,3.242,6.259,0.823,16.042,0.971,24.552


Mean metrics across chronological folds:


,feature_set,n,rmse,bias,nll,coverage_80,width_80,coverage_95,width_95
0,usage_minutes,50392,6.279,-0.689,3.204,0.842,15.560,0.964,23.801
1,attempt_minutes,50392,6.278,-0.690,3.204,0.843,15.560,0.964,23.809
2,both_trends,50392,6.278,-0.689,3.204,0.843,15.561,0.964,23.809


Prior-SD sensitivity on development folds (held out latest test season 2026-27):


,prior_sd_scale,n,rmse,bias,nll,coverage_80,coverage_95
0,0.5,48623,6.136,-0.108,3.186,0.853,0.961
1,1.0,48623,6.136,-0.108,3.186,0.853,0.961
2,2.0,48623,6.136,-0.108,3.186,0.853,0.961


### Verify posterior algebra and standardized vs original-unit priors


In [13]:
from scipy.linalg import cho_factor, cho_solve


# 1) gamma1=0 recovers shared-variance calculations at matching parameters.
train_verify = eligible.sample(
    n=min(2500, len(eligible)),
    random_state=SEED,
).sort_values(["player_id", "game_date"])


shared_fit = fit_points_model(
    train_verify,
    variance_mode="shared",
    context_cols=CONTEXT_FEATURE_COLS,
)
matched = dict(shared_fit)
matched["gamma1"] = 0.0


assert np.allclose(
    observation_variance(
        np.array([1.0, 12.0, 28.0]),
        shared_fit["gamma0"],
        0.0,
        shared_fit["log_b_center"],
    ),
    np.full(3, shared_fit["noise_var"]),
)


shared_pred = predict_points(train_verify.head(20), shared_fit)
matched_pred = predict_points(train_verify.head(20), matched)
assert np.allclose(shared_pred["mean"], matched_pred["mean"], rtol=1e-12, atol=1e-12)
assert np.allclose(
    shared_pred["variance"], matched_pred["variance"], rtol=1e-12, atol=1e-12
)
assert np.allclose(
    matched_pred["observation_variance"],
    np.full(20, shared_fit["noise_var"]),
    rtol=1e-12,
    atol=1e-12,
)


print("gamma1=0 recovery checks passed at matching parameter values.")


# 2) Small direct Gaussian with unequal observation variances.
rng = np.random.default_rng(11)
n_players, games, p = 3, 4, 2
design = rng.normal(size=(n_players * games, p))
codes = np.repeat(np.arange(n_players), games)
target = rng.normal(size=n_players * games)
sigma2 = np.exp(rng.uniform(1.0, 3.0, size=n_players * games))
player_var = 1.3
prior_sd = np.array([3.0, 2.0])
prior_precision = np.diag(1.0 / prior_sd**2)


moments = shared_noise_posterior_moments(
    design=design,
    target=target,
    codes=codes,
    sigma2=sigma2,
    player_var=player_var,
    prior_sd=prior_sd,
    prior_precision=prior_precision,
)


dim = p + n_players
precision = np.zeros((dim, dim))
precision[:p, :p] = prior_precision
for player in range(n_players):
    precision[p + player, p + player] = 1.0 / player_var


rhs = np.zeros(dim)
for i in range(len(target)):
    weight = 1.0 / sigma2[i]
    row = np.zeros(dim)
    row[:p] = design[i]
    row[p + codes[i]] = 1.0
    precision += weight * np.outer(row, row)
    rhs += weight * target[i] * row


chol = cho_factor(precision, lower=True)
mean = cho_solve(chol, rhs)
cov = cho_solve(chol, np.eye(dim))


assert np.allclose(moments["beta"], mean[:p], rtol=1e-10, atol=1e-10)
assert np.allclose(moments["beta_cov"], cov[:p, :p], rtol=1e-10, atol=1e-10)
assert np.allclose(moments["intercept"], mean[p:], rtol=1e-10, atol=1e-10)
assert np.allclose(
    moments["intercept_marginal_var"],
    np.diag(cov[p:, p:]),
    rtol=1e-10,
    atol=1e-10,
)


print("Unequal-variance posterior algebra checks passed.")


# 3) Standardized vs original-unit parameterization equivalence.
std_fit = fit_points_model(
    train_verify,
    variance_mode="shared",
    context_cols=CONTEXT_FEATURE_COLS,
    standardize=True,
)
raw_fit = fit_points_model(
    train_verify,
    variance_mode="shared",
    context_cols=CONTEXT_FEATURE_COLS,
    standardize=False,
    init_gamma0=np.log(std_fit["noise_var"]),
    init_player_var=max(float(std_fit["player_var"]), 1e-8),
)


std_pred = predict_points(train_verify.head(30), std_fit)
raw_pred = predict_points(train_verify.head(30), raw_fit)
conversion = np.r_[1.0, std_fit["scale"]]


assert np.allclose(std_pred["mean"], raw_pred["mean"], rtol=1e-8, atol=1e-8)
assert np.allclose(std_pred["variance"], raw_pred["variance"], rtol=1e-8, atol=1e-8)
assert np.allclose(std_fit["beta"] / conversion, raw_fit["beta"], rtol=1e-8, atol=1e-8)
assert np.allclose(std_fit["prior_sd"], raw_fit["prior_sd"] * conversion)


print(
    "Standardized and original-unit parameterizations "
    "produce equivalent predictions and priors."
)


gamma1=0 recovery checks passed at matching parameter values.
Unequal-variance posterior algebra checks passed.
Standardized and original-unit parameterizations produce equivalent predictions and priors.


### Live betting board

Build pregame features for every player on the newest points slate, predict with the fitted Bayesian model using RotoWire starting flags, and showcase the highest EV bets across books.


In [14]:
from scipy.stats import norm

from models.shared.market import american_to_decimal, no_vig_over
from src.live.predict_points import lineups_on, pinnacle_games_on
from src.live.predict_points_bayes import (
    build_bayesian_live_features,
    choose_live_slate_date,
)
from src.models.points_empirical_bayes import (
    CONTEXT_COLS,
    WINDOW_COLS,
    fit_points_model,
    predict_points,
)


TOP_N = 25
MIN_EV = 0.0


# Prefer a market date that already has a Pinnacle team snapshot.
day = choose_live_slate_date(markets["slate_date"])
slate_markets = markets.loc[
    pd.to_datetime(markets["slate_date"], utc=True).dt.strftime("%Y-%m-%d").eq(day)
].copy()


if slate_markets.empty:
    raise ValueError(f"No points markets for chosen slate date {day}.")


players = slate_markets["player_name"].drop_duplicates().tolist()
games = pinnacle_games_on(day)
lineups = lineups_on(day)


print(f"slate {day}")
print(f"points lines {len(slate_markets):,} across {slate_markets['book'].nunique()} books")
print(f"unique players {len(players):,}")
print(f"pinnacle team games {len(games):,}; rotowire teams {len(lineups):,}")


if not games:
    raise ValueError(
        f"No Pinnacle team markets for {day}. "
        "Run the NBA Pinnacle scraper before pricing the board."
    )


if not lineups:
    print(
        "Warning: no RotoWire lineup snapshot for this date; "
        "starting flags will be 0 unless a projected page is available."
    )




# Fit on all historically eligible rows, then score the live slate.
fitted = fit_points_model(
    eligible,
    variance_mode="baseline",
    context_cols=CONTEXT_FEATURE_COLS,
)


live = build_bayesian_live_features(
    hist,
    players,
    day,
    games=games,
    lineups=lineups,
    context_cols=CONTEXT_FEATURE_COLS,
)


print(
    f"live feature rows {len(live):,} "
    f"(skipped {len(players) - len(live)} unmatched/incomplete)"
)
display(
    live.loc[
        :,
        [
            "player_name",
            "team_abbreviation",
            "opp_abbreviation",
            "is_home",
            "starting",
            "starting_source",
            "start_rate",
            "starter_surprise",
            "opp_def_rating_prior",
            "team_spread_pregame",
            "game_total_pregame",
            "implied_team_total_pregame",
            "usage_trend",
            "minutes_trend",
            "fga_trend",
            "fta_trend",
            "p5",
            "p10",
            "p20",
        ],
    ].round(3)
)




# Known lineup: condition on starting=1/0 rather than a role mixture.
prediction = predict_points(
    live.reset_index(drop=True),
    fitted,
    starter_value=live["starting"].to_numpy(dtype=float),
)


pred = live.copy()
pred["mean"] = prediction["mean"]
pred["predictive_sd"] = np.sqrt(prediction["variance"])
pred["baseline"] = prediction["baseline"]




def continuum_side_probs(mean, variance, line):
    """Integer continuity correction matching the notebook threshold helper."""
    mean = np.asarray(mean, dtype=float)
    sd = np.sqrt(np.asarray(variance, dtype=float))
    line = np.asarray(line, dtype=float)
    p_over = norm.sf(np.floor(line) + 0.5, loc=mean, scale=sd)
    p_under = norm.cdf(np.ceil(line) - 0.5, loc=mean, scale=sd)
    p_push = np.maximum(0.0, 1.0 - p_over - p_under)
    return p_over, p_push, p_under




pred_board = pred.reset_index().rename(columns={"index": "board_name"})
feature_cols = [
    "board_name",
    "team_abbreviation",
    "opp_abbreviation",
    "is_home",
    "starting",
    "starting_source",
    "start_rate",
    "opp_def_rating_prior",
    "team_spread_pregame",
    "game_total_pregame",
    "implied_team_total_pregame",
    "mean",
    "predictive_sd",
    "baseline",
]
board = slate_markets.merge(
    pred_board.loc[:, feature_cols],
    left_on="player_name",
    right_on="board_name",
    how="inner",
    validate="many_to_one",
)


p_over, p_push, p_under = continuum_side_probs(
    board["mean"],
    board["predictive_sd"] ** 2,
    board["line"],
)
board["p_over"] = p_over
board["p_push"] = p_push
board["p_under"] = p_under
board["no_vig_imp_prob_over"] = no_vig_over(
    board["over_american"], board["under_american"]
)
board["no_vig_imp_prob_under"] = 1.0 - board["no_vig_imp_prob_over"]


over_profit = american_to_decimal(board["over_american"]) - 1.0
under_profit = american_to_decimal(board["under_american"]) - 1.0
board["ev_over"] = board["p_over"] * over_profit - board["p_under"]
board["ev_under"] = board["p_under"] * under_profit - board["p_over"]


print(
    f"priced lines {len(board):,} | "
    "p_over+p_push+p_under==1:",
    bool(np.allclose(board["p_over"] + board["p_push"] + board["p_under"], 1.0)),
)




overs = board.assign(side="over", ev=board["ev_over"], american=board["over_american"])
unders = board.assign(side="under", ev=board["ev_under"], american=board["under_american"])
bets = pd.concat([overs, unders], ignore_index=True)


top_ev = (
    bets.loc[bets["ev"].gt(MIN_EV)]
    .sort_values("ev", ascending=False)
    .head(TOP_N)
    .loc[
        :,
        [
            "player_name",
            "team_abbreviation",
            "opp_abbreviation",
            "starting",
            "starting_source",
            "book",
            "side",
            "line",
            "american",
            "mean",
            "predictive_sd",
            "p_over",
            "p_under",
            "p_push",
            "ev",
            "no_vig_imp_prob_over",
            "team_spread_pregame",
            "game_total_pregame",
            "opp_def_rating_prior",
        ],
    ]
)


print(f"Top {TOP_N} positive-EV bets (MIN_EV={MIN_EV:.0%}):")
display(top_ev.round(4))


print("Edge vs no-vig market by book (mean EV of best side per line):")
best_side = bets.loc[bets.groupby(["player_name", "book", "line"])["ev"].idxmax()]
display(
    best_side.groupby("book")
    .agg(
        n=("ev", "size"),
        mean_best_ev=("ev", "mean"),
        pct_positive=("ev", lambda s: float((s > 0).mean())),
        mean_abs_edge=(
            "ev",
            lambda s: float(np.mean(np.abs(s))),
        ),
    )
    .round(4)
)


print(
    "Global window weights:",
    dict(zip(WINDOW_COLS, np.round(fitted["weights"], 3))),
)
print(
    "player-intercept prior sd:"
    f" {np.sqrt(fitted['player_var']):.3f}; "
    f"gamma0={fitted['gamma0']:.3f}, gamma1={fitted['gamma1']:.3f}"
)
display(pd.DataFrame({
    "coefficient": ["level", *fitted["context_cols"]],
    "mean_original": fitted["beta_original"],
    "posterior_sd_original": np.sqrt(np.diag(fitted["beta_cov_original"])),
}).round(3))


slate 2026-11-01
points lines 210 across 2 books
unique players 117
pinnacle team games 88; rotowire teams 8
live feature rows 109 (skipped 8 unmatched/incomplete)


,player_name,team_abbreviation,opp_abbreviation,is_home,starting,starting_source,start_rate,starter_surprise,opp_def_rating_prior,team_spread_pregame,game_total_pregame,implied_team_total_pregame,usage_trend,minutes_trend,fga_trend,fta_trend,p5,p10,p20
board_name,,,,,,,,,,,,,,,,,,,
Shai Gilgeous-Alexander,Shai Gilgeous-Alexander,OKC,CLE,0.0,1.0,start_rate,1.0,0.0,114.21,4.5,228.0,111.75,-0.765,-0.296,1.15,-2.45,24.8,24.9,28.05
Donovan Mitchell,Donovan Mitchell,CLE,OKC,1.0,1.0,start_rate,1.0,0.0,112.00,-4.5,228.0,116.25,-3.615,-0.826,-2.40,-1.20,20.8,24.0,24.40
James Harden,James Harden,CLE,OKC,1.0,1.0,start_rate,1.0,0.0,112.00,-4.5,228.0,116.25,-2.530,-0.228,-0.65,-0.20,16.8,18.8,19.60
Ajay Mitchell,Ajay Mitchell,OKC,CLE,0.0,1.0,start_rate,0.5,0.5,114.21,4.5,228.0,111.75,5.125,1.219,3.40,0.85,19.0,13.2,13.35
Evan Mobley,Evan Mobley,CLE,OKC,1.0,1.0,start_rate,1.0,0.0,112.00,-4.5,228.0,116.25,-1.955,1.790,-0.70,0.15,20.0,18.6,19.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Dennis Schroder,Dennis Schröder,CHO,DET,1.0,0.0,start_rate,0.4,-0.4,107.61,4.0,209.0,102.50,1.610,10.617,4.50,0.00,11.8,11.6,8.85
Nikola Jokic,Nikola Jokić,DEN,POR,1.0,1.0,start_rate,1.0,0.0,110.00,-4.0,231.0,117.50,-0.110,1.098,0.35,-1.05,21.2,23.3,23.65
Isaiah Collier,Isaiah Collier,UTA,SAS,0.0,1.0,start_rate,0.5,0.5,110.50,16.0,231.0,107.50,-2.635,-5.867,-4.45,-0.20,7.6,10.6,13.70


priced lines 197 | p_over+p_push+p_under==1: True
Top 25 positive-EV bets (MIN_EV=0%):


,player_name,team_abbreviation,opp_abbreviation,starting,starting_source,book,side,line,american,mean,predictive_sd,p_over,p_under,p_push,ev,no_vig_imp_prob_over,team_spread_pregame,game_total_pregame,opp_def_rating_prior
89,Brice Sensabaugh,UTA,SAS,1.0,start_rate,prizepicks,over,9.5,-119.0,19.0815,7.1715,0.9092,0.0908,0.0,0.6733,0.5000,16.0,231.0,110.50
178,Brice Sensabaugh,UTA,SAS,1.0,start_rate,underdog,over,9.5,-124.0,19.0815,7.1715,0.9092,0.0908,0.0,0.6425,0.5267,16.0,231.0,110.50
172,Saddiq Bey,NOP,MEM,1.0,start_rate,underdog,over,12.5,100.0,18.5699,7.1928,0.8006,0.1994,0.0,0.6013,0.4764,6.0,236.5,114.87
183,Isaiah Collier,UTA,SAS,1.0,start_rate,underdog,over,6.5,-125.0,14.0347,6.2631,0.8855,0.1145,0.0,0.5939,0.5288,16.0,231.0,110.50
352,Jared McCain,OKC,CLE,0.0,start_rate,underdog,under,14.5,-104.0,9.4043,5.8415,0.1915,0.8085,0.0,0.5859,0.5169,4.5,228.0,114.21
131,GG Jackson,MEM,NOP,1.0,start_rate,underdog,over,9.5,-107.0,14.2875,6.4786,0.7700,0.2300,0.0,0.4897,0.4915,-6.0,236.5,113.99
202,Jared McCain,OKC,CLE,0.0,start_rate,prizepicks,under,14.5,-119.0,9.4043,5.8415,0.1915,0.8085,0.0,0.4879,0.5000,4.5,228.0,114.21
125,Jeremiah Fears,NOP,MEM,1.0,start_rate,underdog,over,13.5,-114.0,19.3716,7.2119,0.7922,0.2078,0.0,0.4872,0.5064,6.0,236.5,114.87
69,Saddiq Bey,NOP,MEM,1.0,start_rate,prizepicks,over,12.5,-119.0,18.5699,7.1928,0.8006,0.1994,0.0,0.4734,0.5000,6.0,236.5,114.87
138,Ace Bailey,UTA,SAS,1.0,start_rate,underdog,over,12.5,-106.0,17.3318,7.0392,0.7538,0.2462,0.0,0.4649,0.4883,16.0,231.0,110.50


Edge vs no-vig market by book (mean EV of best side per line):


,n,mean_best_ev,pct_positive,mean_abs_edge
book,,,,
prizepicks,104,0.1259,0.7885,0.1429
underdog,93,0.1841,0.9247,0.1884


Global window weights: {'p5': np.float64(0.25), 'p10': np.float64(0.13), 'p20': np.float64(0.62)}
player-intercept prior sd: 0.000; gamma0=3.505, gamma1=0.786


,coefficient,mean_original,posterior_sd_original
0,level,0.193,0.018
1,starter_surprise,3.238,0.060
2,is_home,0.225,0.038
3,back2back,0.278,0.048
4,opp_def_rating_prior,0.071,0.004
5,team_spread_pregame,0.020,0.003
6,implied_team_total_pregame,-0.029,0.004
7,usage_trend,-0.015,0.007
8,minutes_trend,0.039,0.008
9,fga_trend,-0.060,0.021
